In [ ]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.inspection import permutation_importance

In [ ]:


# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("dataset.csv")
df["date"] = pd.to_datetime(df["date"])

# =========================
# 2. CREATE SEASON COLUMN
# =========================
def assign_season(date):
    year = date.year
    if date.month >= 7:  # campaña empieza en julio
        return f"{year}-{year+1}"
    else:
        return f"{year-1}-{year}"

df["season"] = df["date"].apply(assign_season)

# =========================
# 3. SPLIT
# =========================
train_df = df[df["season"] == "2024-2025"]
test_df  = df[df["season"] == "2025-2026"]

# =========================
# 4. FEATURES
# =========================
features = [
    "lat",
    "lon",
    "temp_mean",
    "precip",
    "humidity"
]

X_train = train_df[features]
y_train = train_df["outbreak"]

X_test = test_df[features]
y_test = test_df["outbreak"]

# =========================
# 5. MODEL
# =========================
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

# =========================
# 6. EVALUATION
# =========================
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, (y_pred_proba > 0.5).astype(int)))

# =========================
# 7. FEATURE IMPORTANCE (MODEL)
# =========================
importance = model.get_booster().get_score(importance_type="gain")

print("\nFeature importance (gain):")
for k, v in sorted(importance.items(), key=lambda x: -x[1]):
    print(f"{k}: {v:.4f}")

# =========================
# 8. PERMUTATION IMPORTANCE (TRAIN)
# =========================
perm_train = permutation_importance(
    model, X_train, y_train,
    n_repeats=10,
    random_state=42,
    scoring="roc_auc"
)

print("\nPermutation importance (TRAIN):")
for i in perm_train.importances_mean.argsort()[::-1]:
    print(f"{features[i]}: {perm_train.importances_mean[i]:.4f}")

# =========================
# 9. PERMUTATION IMPORTANCE (TEST)
# =========================
perm_test = permutation_importance(
    model, X_test, y_test,
    n_repeats=10,
    random_state=42,
    scoring="roc_auc"
)

print("\nPermutation importance (TEST):")
for i in perm_test.importances_mean.argsort()[::-1]:
    print(f"{features[i]}: {perm_test.importances_mean[i]:.4f}")